In [ ]:
!nvidia-smi

: 

: 

In [ ]:
# ============================================================
# PHASE 10 — UNCERTAINTY ESTIMATION
# STEP 10.1 — VERIFY MC DROPOUT BEHAVIOR
# ============================================================

import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# 2. Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ------------------------------------------------------------
# 3. Import model
# ------------------------------------------------------------

from src.models.proposed import MultiScaleSTGCN


# ------------------------------------------------------------
# 4. Configuration
# ------------------------------------------------------------

INPUT_STEPS = 12
OUTPUT_STEPS = 3
NUM_NODES = 207
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = 32
TEMPORAL_KERNELS = [3, 5]
DROPOUT = 0.2


# ------------------------------------------------------------
# 5. Build model
# ------------------------------------------------------------

model = MultiScaleSTGCN(
    num_nodes=NUM_NODES,
    input_steps=INPUT_STEPS,
    output_steps=OUTPUT_STEPS,
    input_channels=INPUT_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    temporal_kernels=TEMPORAL_KERNELS,
    dropout=DROPOUT,
).to(DEVICE)


# ------------------------------------------------------------
# 6. Find checkpoint
# ------------------------------------------------------------

checkpoint_candidates = [
    Path("/content/experiments/proposed/best_model.pt"),
    Path("/content/experiments/proposed/best_model.pth"),
    Path("/content/experiments/proposed/checkpoint.pt"),
    Path("/content/experiments/proposed/checkpoint.pth"),
]

checkpoint_path = None

for path in checkpoint_candidates:
    if path.exists():
        checkpoint_path = path
        break

if checkpoint_path is None:
    raise FileNotFoundError(
        "Could not find Proposed STGCN checkpoint. "
        "Check /content/experiments/proposed/"
    )

print("Checkpoint:", checkpoint_path)


# ------------------------------------------------------------
# 7. Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE
)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        state_dict = checkpoint

else:
    state_dict = checkpoint


model.load_state_dict(state_dict)

print("Checkpoint loaded successfully.")


# ------------------------------------------------------------
# 8. Find dropout layers
# ------------------------------------------------------------

dropout_layers = [
    module
    for module in model.modules()
    if isinstance(module, nn.Dropout)
]

print("\nDropout layers found:", len(dropout_layers))

for i, layer in enumerate(dropout_layers):
    print(
        f"Dropout {i}: "
        f"p={layer.p}"
    )

assert len(dropout_layers) > 0, (
    "No Dropout layers found. "
    "MC Dropout cannot be performed."
)


# ------------------------------------------------------------
# 9. Load graph
# ------------------------------------------------------------

import pickle

ADJACENCY_PATH = Path(
    "/content/data/raw/METR-LA/adj_METR-LA.pkl"
)

with open(ADJACENCY_PATH, "rb") as f:
    sensor_ids, sensor_id_map, adjacency_matrix = pickle.load(
        f,
        encoding="latin1"
    )

A = adjacency_matrix.astype(np.float32)

degree = A.sum(axis=1)

assert np.all(degree > 0)

degree_inv = 1.0 / degree

A_normalized = (
    degree_inv[:, np.newaxis] * A
).astype(np.float32)

A_tensor = torch.from_numpy(
    A_normalized
).to(DEVICE)

print("\nGraph shape:", A_tensor.shape)


# ------------------------------------------------------------
# 10. Load one real test batch
# ------------------------------------------------------------

test_x = np.load(
    "/content/data/processed/METR-LA/test.npz"
)["X"]

test_y = np.load(
    "/content/data/processed/METR-LA/test.npz"
)["Y"]

print("Test X:", test_x.shape)
print("Test Y:", test_y.shape)


# ------------------------------------------------------------
# 11. Prepare one batch
# ------------------------------------------------------------

BATCH_SIZE = 16

x_batch = torch.from_numpy(
    test_x[:BATCH_SIZE]
).float().to(DEVICE)

y_batch = torch.from_numpy(
    test_y[:BATCH_SIZE]
).float().to(DEVICE)


# ------------------------------------------------------------
# 12. Deterministic evaluation
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    pred_1 = model(
        x_batch,
        A_tensor
    )

    pred_2 = model(
        x_batch,
        A_tensor
    )


deterministic_difference = torch.max(
    torch.abs(pred_1 - pred_2)
).item()

print(
    "\nDeterministic evaluation difference:",
    deterministic_difference
)


# ------------------------------------------------------------
# 13. Enable dropout during inference
# ------------------------------------------------------------

model.eval()

for module in model.modules():

    if isinstance(module, nn.Dropout):
        module.train()


# ------------------------------------------------------------
# 14. Two stochastic forward passes
# ------------------------------------------------------------

with torch.no_grad():

    stochastic_pred_1 = model(
        x_batch,
        A_tensor
    )

    stochastic_pred_2 = model(
        x_batch,
        A_tensor
    )


stochastic_difference = torch.max(
    torch.abs(
        stochastic_pred_1 -
        stochastic_pred_2
    )
).item()

print(
    "MC Dropout stochastic difference:",
    stochastic_difference
)


# ------------------------------------------------------------
# 15. Shape verification
# ------------------------------------------------------------

print(
    "\nPrediction shape:",
    stochastic_pred_1.shape
)

assert stochastic_pred_1.shape == (
    BATCH_SIZE,
    OUTPUT_STEPS,
    NUM_NODES,
    1
)

assert stochastic_pred_2.shape == (
    BATCH_SIZE,
    OUTPUT_STEPS,
    NUM_NODES,
    1
)


# ------------------------------------------------------------
# 16. Final checks
# ------------------------------------------------------------

assert deterministic_difference < 1e-7, (
    "Model evaluation is not deterministic."
)

assert stochastic_difference > 0, (
    "MC Dropout did not produce stochastic predictions."
)

assert torch.all(
    torch.isfinite(stochastic_pred_1)
)

assert torch.all(
    torch.isfinite(stochastic_pred_2)
)


print("\nMC Dropout verification PASSED.")

: 

: 